# Min-Cost Max-Flow using Geometric Simplex Method
### Submitted by:
- **Piyush Anand : CS25MTECH12009**
- **Darshanraj Pattanaik : CS25MTECH12002**

---

## 1. Problem Description

We are given a directed network with capacities and per-unit costs defined on each edge. The goal is to solve the **Minimum-Cost Maximum-Flow** problem:

1. Find the **maximum possible flow** from the source node (1) to the sink node (2).
2. Among all flows achieving this value, compute the **minimum total cost**.

The input file is a CSV with exactly four rows:

1. From-vertex  
2. To-vertex  
3. Capacity of the edge  
4. Cost per unit flow  

Each column corresponds to one directed edge.

---



## 1. Imports and Global Parameters

This section initializes all required Python libraries and defines
constants used throughout the implementation. The solver uses
NumPy for numerical computation, Pandas for reading CSV input,
and SciPy's `null_space` for computing nullspaces in the geometric
simplex method.

We also define global constants such as:
- `SRC` and `SINK` (source and destination vertex IDs)
- `TOL` (numerical tolerance)
These are used globally by all subsequent modules.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.linalg import null_space

SRC = 1
SINK = 2
TOL = 1e-9
MAX_TESTCASES = 15

## 2. Reading the Graph Input

The input CSV contains four rows:
1. From-vertex for each edge  
2. To-vertex for each edge  
3. Capacity  
4. Cost per unit flow  

This module parses the CSV file and produces Python lists of:
- `u_list` (from nodes)
- `v_list` (to nodes)
- `cap_list` (capacities)
- `cost_list` (costs)
- `max_node` (largest vertex index)

These are used directly to construct the LP in later stages.


In [ ]:
def read_graph_csv(filename):
    df = pd.read_csv(filename, header=None)
    if df.shape[0] < 4:
        raise ValueError("CSV must contain 4 rows: from, to, capacity, cost")
    u_list = df.iloc[0].astype(int).tolist()
    v_list = df.iloc[1].astype(int).tolist()
    cap_list = df.iloc[2].astype(float).tolist()
    cost_list = df.iloc[3].astype(float).tolist()
    max_node = max(max(u_list), max(v_list))
    return u_list, v_list, cap_list, cost_list, max_node


## 3. LP Construction from the Graph

This section converts the flow network into an inequality-based LP:

- Flow conservation for internal vertices  
- Capacity constraints  
- Non-negativity constraints  
- Objective function for:
  - Phase A (maximize flow → minimize negative flow)
  - Phase B (minimize total cost)

The LP takes the standard form:
    A x <= b
where `x` is the vector of flows on edges.


In [ ]:
def build_lp_from_graph(u_list, v_list, cap_list, cost_list, max_node, src=1, sink=2):
    m_edges = len(u_list)
    rows, rhs = [], []

    # Flow conservation constraints (two inequalities each)
    for node in range(1, max_node + 1):
        if node in (src, sink):
            continue
        row = np.zeros(m_edges)
        for j in range(m_edges):
            if u_list[j] == node: row[j] += 1
            if v_list[j] == node: row[j] -= 1
        rows.append(row); rhs.append(0)
        rows.append(-row); rhs.append(0)

    # Bounds on each variable
    for j in range(m_edges):
        up = np.zeros(m_edges); up[j] = 1
        rows.append(up); rhs.append(cap_list[j])
        lo = np.zeros(m_edges); lo[j] = -1
        rows.append(lo); rhs.append(0)

    A = np.vstack(rows)
    b = np.array(rhs, dtype=float)

    # Phase A objective: maximize flow = minimize negative flow
    objA = np.zeros(m_edges)
    for j in range(m_edges):
        if u_list[j] == src: objA[j] += 1
        if v_list[j] == src: objA[j] -= 1
    c_phaseA = -objA

    # Phase B objective: minimize cost
    c_phaseB = np.array(cost_list)

    return A, b, c_phaseA, c_phaseB


## 4. Phase I Auxiliary LP (Finding a Feasible Point)

The geometric simplex method requires an initial feasible vertex.
To obtain one, we construct an auxiliary LP with an extra variable `t`,
which measures violation of constraints.

We minimize `t` subject to:
    A x <= b + t
    t >= 0

If the optimal `t` equals zero, a feasible solution exists.
Otherwise, the LP is infeasible.

This section builds the augmented LP and the starting point.


In [ ]:
def build_phase1_lp(A, b):
    m, n = A.shape
    A1 = np.hstack([A, -np.ones((m,1))])
    b1 = b.copy()
    A2 = np.hstack([np.zeros((1,n)), -np.ones((1,1))])
    b2 = np.zeros(1)
    A_hat = np.vstack([A1, A2])
    b_hat = np.concatenate([b1, b2])
    c_hat = np.zeros(n+1); c_hat[-1] = 1
    y0 = np.zeros(n+1)
    y0[-1] = max(0, -min(b))
    return A_hat, b_hat, c_hat, y0


## 5. Moving to a Vertex

The geometric simplex method operates only on vertices.  
Given a feasible point, this module moves to a feasible vertex by:

1. Identifying tight constraints
2. Computing a nullspace direction
3. Performing a ratio test
4. Stepping to the nearest blocking constraint

This guarantees that the method starts Phase II at a valid vertex.


In [ ]:
def move_feasible_to_vertex(A, b, x, n_dim, tol=TOL, max_hops=10000):
    for _ in range(max_hops):
        r = A @ x - b
        tight = np.where(np.abs(r) < tol)[0]
        A_tight = A[tight,:] if len(tight)>0 else np.zeros((0,n_dim))
        if A_tight.size>0 and np.linalg.matrix_rank(A_tight)==n_dim:
            return x, [x]
        if A_tight.size==0:
            u = np.random.randn(n_dim)
        else:
            ns = null_space(A_tight)
            u = ns[:,0] if ns.size>0 else np.random.randn(n_dim)

        loose = np.where(np.abs(r)>=tol)[0]
        alphas = []
        for j in loose:
            denom = A[j]@u
            if denom>tol:
                t = (b[j]-A[j]@x)/denom
                if t>tol:
                    alphas.append(t)
        if not alphas: return x, [x]
        x = x + min(alphas)*u
    return x, [x]


## 6. Geometric Simplex Solver (Phase II)

This section implements the complete pivoting mechanism:

- Basis construction from tight constraints
- Computation of reduced-cost directions
- Selection of entering direction
- Ratio test to find blocking constraint
- Pivoting step
- Iteration logging

The solver minimizes a linear objective under inequality constraints.

In [ ]:
def pick_full_rank_subset(A_rows, want_cols):
    idx=[]
    for i in range(A_rows.shape[0]):
        T = idx+[i]
        if np.linalg.matrix_rank(A_rows[T,:])==len(T):
            idx=T
            if len(idx)==want_cols:
                return idx
    return None

def geometric_simplex_trace_minimal(A, b, c, x0, tol=TOL, max_iters=1000):
    m,n = A.shape
    def tight_idx(x): return np.where(np.abs(A@x-b)<=tol)[0]

    # Ensure we start at a vertex
    B=None
    T=tight_idx(x0)
    if len(T)>=n:
        L=pick_full_rank_subset(A[T,:],n)
        if L is not None: B=[T[i] for i in L]
    if B is None:
        x0,_ = move_feasible_to_vertex(A,b,x0,n)
        T=tight_idx(x0)
        L=pick_full_rank_subset(A[T,:],n)
        B=[T[i] for i in L]

    x=x0.copy()

    print("Iter |    Obj    | EntDir | Leave | Block |  Step")
    print("--------------------------------------------------------")

    for it in range(1,max_iters+1):
        A_B=A[B,:]
        A_B_inv=np.linalg.inv(A_B)
        V=-A_B_inv
        dirs=[V[:,j] for j in range(n)]
        gains=[c@v for v in dirs]
        obj=float(c@x)

        if all(g>=-tol for g in gains):
            print("--------------------------------------------------------")
            return x,obj,[x]

        e=int(np.argmin(gains))
        v=dirs[e]

        # Ratio test
        blocks=[]
        for j in range(m):
            if j in B: continue
            denom=A[j]@v
            if denom>tol:
                t=(b[j]-A[j]@x)/denom
                if t>=0: blocks.append((t,j))
        if not blocks:
            print(f"{it} | {obj:10.2f} | {e} | --- | --- | unbounded")
            return x,float('inf'),[x]

        t_star,block=min(blocks,key=lambda z:z[0])
        leave=B[e]

        if t_star>tol:
            x=x+t_star*v

        if block not in B:
            B[e]=block

        print(f"{it:4d} | {obj:10.2f} | {e:7d} | {leave:5d} | {block:5d} | {t_star:6.2f}")

    return x,float(c@x),[x]


## 7. Two-Phase Simplex Driver

This function performs:

1. Phase I  to find a feasible flow.
2. Phase II to optimize:
   - Either maximizing flow (Phase A LP)
   - Or minimizing cost (Phase B LP)

It combines all previously defined modules and prints iteration logs.

In [ ]:
def geometric_simplex_full_minimal(A, b, c):
    m,n=A.shape

    # Phase I
    A_hat,b_hat,c_hat,y0=build_phase1_lp(A,b)
    y_vertex,_ = move_feasible_to_vertex(A_hat,b_hat,y0,n+1)

    import sys
    old=sys.stdout
    sys.stdout=open(os.devnull,'w')
    y_star,obj_aux,_ = geometric_simplex_trace_minimal(A_hat,b_hat,c_hat,y_vertex)
    sys.stdout.close()
    sys.stdout=old

    if y_star[-1]>1e-9:
        raise RuntimeError("LP infeasible")

    x_feasible=y_star[:-1]
    x_vertex,_ = move_feasible_to_vertex(A,b,x_feasible,n)

    return geometric_simplex_trace_minimal(A,b,c,x_vertex)


## 8. Running the Solver on a Testcase

This final section loads a testcase CSV, runs:

- Phase A LP (maximum flow)
- Phase B LP (minimum cost for that flow)

Finally, it prints the full flow distribution on all edges.

In [ ]:
def run_single_file_minimal(filename):
    u,v,cap,cost,max_node = read_graph_csv(filename)
    A,b,cA,cB = build_lp_from_graph(u,v,cap,cost,max_node,SRC,SINK)

    print(f"\n===== {filename} =====")
    xA,objA,_ = geometric_simplex_full_minimal(A,b,cA)
    F_star=-objA
    print(f"\nMax flow (F*) = {F_star:.2f}")

    m_base,n_vars = A.shape
    row_src=np.zeros(n_vars)
    for j in range(n_vars):
        if u[j]==SRC: row_src[j]+=1
        if v[j]==SRC: row_src[j]-=1
    A2=np.vstack([A,row_src,-row_src])
    b2=np.concatenate([b,[F_star,-F_star]])

    xB,objB,_ = geometric_simplex_full_minimal(A2,b2,cB)
    print(f"\nMin cost = {objB:.2f}")

    print("\nFinal flows:")
    print("------------------------------------")
    for j in range(len(u)):
        if abs(xB[j])>1e-9:
            print(f" e{j}: {u[j]} -> {v[j]}  | flow={xB[j]:.2f}, cost={cost[j]:.2f}")
    print("------------------------------------")


## Testing


In [ ]:
import pandas as pd

data = [
    [1,1,3,4,3,4,5,6,5],   # from
    [3,4,5,5,6,6,2,2,6],   # to
    [15,10,10,5,5,10,10,10,4],  # capacity
    [4,2,1,3,2,5,3,1,1]    # cost
]

df = pd.DataFrame(data)
df.to_csv("testcase_1.csv", header=False, index=False)

In [ ]:
if __name__ == "__main__":
    run_single_file_minimal("Testcase1.csv")

TypeError: 'NoneType' object is not iterable